# Messidor-2 external evaluation

Evaluates the saved raw-image and cropped-training ResNet-18 checkpoints on the same processed Messidor-2 images. Labels are aligned by image identifier.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os,glob,cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from tqdm import tqdm
BASE_DIR='/content/drive/MyDrive/Fundus_Artifact_Project'
MESSIDOR_DIR=os.path.join(BASE_DIR,'Messidor_2')
PRE_DIR=os.path.join(MESSIDOR_DIR,'my_preprocessed')
SAVE_DIR=os.path.join(BASE_DIR,'Results','dr_artifact_model')
df=pd.read_csv(os.path.join(MESSIDOR_DIR,'grades.csv')).dropna(subset=['adjudicated_dr_grade']).copy()
df['adjudicated_dr_grade']=df['adjudicated_dr_grade'].astype(int)
id_to_y={os.path.splitext(str(r['image_id']))[0]: int(r['adjudicated_dr_grade']>0) for _,r in df.iterrows()}
paths=[]; y=[]
for fp in sorted(glob.glob(os.path.join(PRE_DIR,'*.png'))):
    stem=os.path.splitext(os.path.basename(fp))[0]
    if stem in id_to_y: paths.append(fp); y.append(id_to_y[stem])
y=np.asarray(y,dtype=int)
print('Matched external images:',len(paths),'positive prevalence:',y.mean() if len(y) else float('nan'))


In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def build_model(path):
    m=models.resnet18(weights=None); m.fc=nn.Linear(m.fc.in_features,2)
    m.load_state_dict(torch.load(path,map_location=device)); return m.to(device).eval()
RAW_CKPT=os.path.join(BASE_DIR,'Results','raw_image_model','dr_classifier_resnet18.pt')
CLEAN_CKPT=os.path.join(BASE_DIR,'Results','dr_artifact_model','dr_clean_resnet18.pt')
raw_model=build_model(RAW_CKPT); clean_model=build_model(CLEAN_CKPT)


In [ ]:
mean=torch.tensor([0.485,0.456,0.406]).view(3,1,1); std=torch.tensor([0.229,0.224,0.225]).view(3,1,1)
def load_tensor(fp):
    img=cv2.imread(fp,cv2.IMREAD_COLOR)
    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB); img=cv2.resize(img,(224,224),interpolation=cv2.INTER_AREA)
    t=torch.from_numpy(img).permute(2,0,1).float()/255.0
    return (t-mean)/std
def infer(model,paths,batch_size=64):
    scores=[]; preds=[]
    with torch.no_grad():
        for i in tqdm(range(0,len(paths),batch_size)):
            x=torch.stack([load_tensor(p) for p in paths[i:i+batch_size]]).to(device)
            out=model(x); prob=F.softmax(out,dim=1)
            # ImageFolder sorts the training folders alphabetically: dr=0, no_dr=1.
            dr_score=prob[:,0]
            scores.extend(dr_score.cpu().numpy()); preds.extend((dr_score>=0.5).long().cpu().numpy())
    return np.asarray(scores),np.asarray(preds)
raw_scores,raw_preds=infer(raw_model,paths); clean_scores,clean_preds=infer(clean_model,paths)


In [ ]:
from sklearn.metrics import roc_auc_score,roc_curve,confusion_matrix,accuracy_score
import matplotlib.pyplot as plt
def evaluate(name,scores,preds):
    auc=roc_auc_score(y,scores); acc=accuracy_score(y,preds); cm=confusion_matrix(y,preds,labels=[0,1])
    print(name,'accuracy=',round(acc,4),'AUC=',round(auc,4),'confusion matrix=\n',cm)
    return {'model':name,'accuracy':acc,'roc_auc':auc,'tn':int(cm[0,0]),'fp':int(cm[0,1]),'fn':int(cm[1,0]),'tp':int(cm[1,1])}
rows=[evaluate('raw',raw_scores,raw_preds),evaluate('artifact_aware',clean_scores,clean_preds)]
pd.DataFrame(rows).to_csv(os.path.join(SAVE_DIR,'messidor2_external_metrics.csv'),index=False)
plt.figure(figsize=(6,5))
for name,scores in [('Raw',raw_scores),('Artifact-aware',clean_scores)]:
    fpr,tpr,_=roc_curve(y,scores); plt.plot(fpr,tpr,label=f'{name} AUC={roc_auc_score(y,scores):.3f}')
plt.plot([0,1],[0,1],'--'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'messidor2_external_roc.png'),dpi=300,bbox_inches='tight'); plt.show()


## Class mapping

The DR probability is taken directly from the training class order (`dr = 0`, `no_dr = 1`). Scores are not inverted after evaluation.
